<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/main/notebooks/workbook_marvin_training_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 – Training mit Wikitext (einfach)

**Autor:** Marvin Ebi  
**Ziel:** Einfaches Training wie auf den Folien: Wikitext aus Hugging Face laden, mit `BPEDataset` Token-Fenster erzeugen, per PyTorch `DataLoader` trainieren und den Loss auswerten.

In Colab installiert `uv` die Notebook-Pakete in den aktiven Kernel. Eine separate venv wäre nicht automatisch der Notebook-Kernel und könnte Colabs GPU-PyTorch umgehen. Der Modell-Checkpoint wird unter `checkpoints/` gespeichert; dieser Ordner ist von Git ausgeschlossen.


In [2]:
from pathlib import Path
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path('/content/INFM_SOTA2')
    if not REPO_DIR.exists():
        import subprocess
        subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/Timbo3399/INFM_SOTA2.git', str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)

    # Keep Colab's GPU-enabled torch; install notebook dependencies in its active kernel.
    !python -m pip install -q uv
    !uv pip install --system --no-deps -e .
    !uv pip install --system "datasets==5.1.0" "huggingface-hub==2.4.0" "fsspec==2026.7.0" "tiktoken==0.14.0" "wandb==0.28.1" "nbformat==5.11.1"
else:
    REPO_DIR = next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / 'pyproject.toml').is_file() and (p / 'src' / 'infm_sota2').is_dir()),
        None,
    )
    if REPO_DIR is None:
        raise FileNotFoundError('Projektwurzel nicht gefunden. Notebook im Repo starten.')
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Projektpfad: {REPO_DIR}')


Projektpfad: /home/marvin/Masterstudium/Aktuelle Methoden des Maschinellen Lernens II/INFM_SOTA2


## 1. Wikitext aus Hugging Face laden

Wir laden nur einen kleinen Ausschnitt, damit das Notebook schnell läuft.


In [3]:
from datasets import load_dataset

train_hf = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train[:200]')
val_hf = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='validation[:50]')

print(train_hf)
print(val_hf)


Dataset({
    features: ['text'],
    num_rows: 200
})
Dataset({
    features: ['text'],
    num_rows: 50
})


## 2. BPEDataset und DataLoader bauen

Die Hugging-Face-Zeilen werden zu einem Text zusammengefügt. `BPEDataset` tokenisiert ihn und erzeugt Eingabe-/Zielsequenzen mit einem Token Versatz.


In [4]:
import torch
from torch.utils.data import DataLoader
from infm_sota2.data import BPEDataset

SEQ_LEN = 64
BATCH_SIZE = 4

train_text = "\n".join(text for text in train_hf['text'] if text.strip())
val_text = "\n".join(text for text in val_hf['text'] if text.strip())

train_dataset = BPEDataset(train_text, seq_len=SEQ_LEN, stride=SEQ_LEN)
val_dataset = BPEDataset(val_text, seq_len=SEQ_LEN, stride=SEQ_LEN)

tokenizer = train_dataset.tokenizer
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train windows: {len(train_dataset)}')
print(f'Validation windows: {len(val_dataset)}')

x, y = next(iter(train_loader))
print('Batch shape x:', x.shape)
print('Batch shape y:', y.shape)


Train windows: 184
Validation windows: 34
Batch shape x: torch.Size([4, 64])
Batch shape y: torch.Size([4, 64])


## 3. Modell trainieren, mit W&B verfolgen und speichern

Beim ersten Start meldest du dich mit `wandb.login()` an. In Colab kannst du alternativ `WANDB_API_KEY` unter **Secrets** hinterlegen. Der Key gehört nicht ins Notebook. W&B speichert Trainings- und Validierungs-Loss sowie die verwendeten Hyperparameter.


In [ ]:
from infm_sota2 import GPTModel
import wandb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG = {
    'vocab_size': train_dataset.vocab_size,
    'emb_dim': 64,
    'context_length': SEQ_LEN,
    'n_layers': 2,
    'n_heads': 4,
    'drop_rate': 0.1,
    'qkv_bias': True,
}

LEARNING_RATE = 0.001
EPOCHS = 1
if IN_COLAB:
    from google.colab import userdata
    api_key = userdata.get('WANDB_API_KEY', default=None)
    wandb.login(key=api_key) if api_key else wandb.login()
else:
    wandb.login()
run = wandb.init(
    project='SOTA_2',
    name='first_run',
    config={
        'dataset': 'Salesforce/wikitext-2-raw-v1',
        'train_examples': len(train_dataset),
        'validation_examples': len(val_dataset),
        'batch_size': BATCH_SIZE,
        'sequence_length': SEQ_LEN,
        'epochs': EPOCHS,
        'optimizer': 'AdamW',
        'learning_rate': LEARNING_RATE,
        **CFG,
    },
)

model = GPTModel(CFG).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    count = 0
    for step, (x, y) in enumerate(train_loader):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = loss_fn(logits.reshape(-1, CFG['vocab_size']), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        count += 1

        if step % 5 == 0:
            print(f'epoch={epoch + 1}, step={step}, loss={loss.item():.4f}')
        wandb.log({'train/loss': loss.item(), 'epoch': epoch + 1})

    train_loss = running_loss / max(count, 1)
    print(f'Average training loss: {train_loss:.4f}')

# kurze Validierung
model.eval()
val_loss = 0.0
num_batches = 0
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x)
        loss = loss_fn(logits.reshape(-1, CFG['vocab_size']), y.reshape(-1))
        val_loss += loss.item()
        num_batches += 1

val_loss /= max(num_batches, 1)
print(f'Validation loss: {val_loss:.4f}')
wandb.log({'train/epoch_loss': train_loss, 'validation/loss': val_loss, 'epoch': EPOCHS})

checkpoint_dir = REPO_DIR / 'checkpoints'
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / f'gpt-wikitext-{run.id}.pt'
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': CFG,
    'train_loss': train_loss,
    'validation_loss': val_loss,
}, checkpoint_path)
print(f'Checkpoint gespeichert: {checkpoint_path}')
run.finish()


## 4. Ein kleiner Text-Check

Damit sehen wir, dass das Modell nach dem Training überhaupt etwas erzeugt.


In [12]:
model.eval()
prompt = 'The model is'
prompt_ids = tokenizer.encode(prompt)
prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
with torch.no_grad():
    logits = model(prompt_tensor)
    next_token = logits[:, -1, :].argmax(dim=-1).item()
    print('Nächstes Token:', next_token)
    generated = prompt + tokenizer.decode([next_token])
    print('Generiert:', generated)


Nächstes Token: 25084
Generiert: The model isキ
